# ДЗ10: помогите роботу пройти полосу препятствий / HW10: help a robot navigate obstacles

**Пять небольших функций → один алгоритм Q-learning. CPU, NumPy, RU/EN.**

**RU.** Робот стоит в клетке S. Ему нужно дойти до G, обходя стены. Сначала он ходит почти наугад. За каждый шаг получает −1, за удар в стену −3, за достижение цели +20. Повторяя попытки, робот запоминает, какое движение полезнее в каждой клетке. Мы напишем это по частям, без нейросетей, производных и матричного умножения.

**EN.** A robot starts at S and must reach G without crossing walls. Initially it moves almost randomly. Each ordinary move gives −1, a collision gives −3, reaching the goal gives +20. Across repeated attempts it learns which move is useful in each cell. We build this step by step, without neural networks, derivatives or matrix multiplication.

Это явно заданная **учебная симуляция**, а не реальный датасет и не физический робот. В этой версии по просьбе преподавателя используется полоса препятствий. / This is an explicitly defined **teaching simulation**, not a real dataset or physical robot. This revision uses an obstacle course at the instructor's request.

## Как работать / How to work
1. Выполните подготовку и посмотрите карту. / Run setup and inspect the map.
2. В каждом из пяти блоков: прочитайте материал → выполните пример → заполните функцию → запустите проверку → напишите короткий ответ. / In each of five blocks: read → run the example → complete the function → run the check → write a short answer.
3. Заменяйте `...`, удаляйте `raise NotImplementedError` после завершения функции. Это сообщение означает «задание ещё не заполнено». `assert` проверяет результат: при ошибке прочитайте его сообщение. / Replace `...` and remove `raise NotImplementedError` when the function is complete. It marks unfinished work. An `assert` checks a result; read its message if it fails.
4. Финальные ячейки обучения и рисунка уже даны. Не нужно писать ещё один эксперимент. / Final training and plotting cells are supplied; no additional experiment is required.

| Слово / Term | Значение здесь / Meaning here |
|---|---|
| Состояние / State | Клетка `(row, col)`, например `(0, 0)` / Cell, e.g. `(0, 0)` |
| Действие / Action | 0 вверх, 1 вправо, 2 вниз, 3 влево / 0 up, 1 right, 2 down, 3 left |
| Награда / Reward | Число, которое сообщает среда после хода / Number returned by the environment after a move |
| Эпизод / Episode | Одна попытка от S до G либо до лимита шагов / One attempt from S to G or the step limit |
| `Q[row, col, action]` | Насколько полезным робот считает это действие в этой клетке / Estimated usefulness of that action in that cell |

**Важно / Important:** действие меняет следующую клетку, поэтому это последовательная задача. Q оценивает будущую суммарную награду, а не вероятность. Карта известна функции среды; обучение использует только результат одного сделанного хода. / Actions change the next cell, making this a sequential task. Q estimates future cumulative reward, not probability. The environment knows the map; training uses only the result of the action actually taken.

Версия / Version: `HW10_Robot_Qlearning_v2`. Ранее сданный Wine-вариант сохраняет свою рубрику; повторная сдача из-за смены версии не нужна. / Previously submitted Wine work retains its original rubric; this revision does not require resubmission.

In [ ]:
import numpy as np # Массивы и случайные числа / Arrays and random numbers.
import matplotlib.pyplot as plt # Рисунок карты и маршрута / Map and route plotting.
STUDENT_NAME = '' # Ваше имя / Your name.
STUDENT_GROUP = '' # Ваша группа / Your group.
ASSIGNMENT_VERSION = 'HW10_Robot_Qlearning_v2' # Отличать от старого Wine / Distinguish from old Wine version.
ROWS, COLS = 5, 6 # Размер карты / Map size.
START, GOAL = (0, 0), (4, 5) # Начало и цель / Start and goal.
WALLS = {(0, 2), (1, 2), (2, 2), (2, 4), (3, 4), (4, 1)} # Непроходимые клетки / Blocked cells.
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)] # Изменения row/col: вверх, вправо, вниз, влево / Up, right, down, left offsets.
ACTION_NAMES = ['up / вверх', 'right / вправо', 'down / вниз', 'left / влево'] # Подписи действий / Action labels.
def draw_map(path=None): # Готовая функция, редактировать не нужно / Supplied plotting helper.
    grid = np.zeros((ROWS, COLS)) # Свободные клетки / Free cells.
    for row, col in WALLS: # Перебрать стены / Visit walls.
        grid[row, col] = 1 # Пометить стену / Mark a wall.
    fig, ax = plt.subplots(figsize=(7, 5)) # Создать рисунок / Create a figure.
    ax.imshow(grid, cmap='Greys', vmin=0, vmax=1) # Чёрный — стена / Black means wall.
    ax.text(START[1], START[0], 'S', ha='center', va='center', color='green', fontsize=18) # Начало / Start.
    ax.text(GOAL[1], GOAL[0], 'G', ha='center', va='center', color='green', fontsize=18) # Цель / Goal.
    if path is not None: # Рисовать маршрут, если передан / Draw an optional route.
        ax.plot([p[1] for p in path], [p[0] for p in path], 'o-', color='tab:blue', alpha=0.6) # Соединить клетки / Connect visited cells.
    ax.set_xticks(range(COLS)); ax.set_yticks(range(ROWS)) # Координаты / Coordinates.
    ax.set_xlabel('col / столбец'); ax.set_ylabel('row / строка') # Названия осей / Axis names.
    ax.set_title('Robot obstacle course / Полоса препятствий') # Заголовок / Title.
    plt.show() # Показать рисунок / Display figure.
draw_map() # Посмотреть задачу / Inspect the task.

## 1. Один ход робота / One robot move

**Зачем / Why:** функция `step` — правила нашего мира. Она получает текущую клетку и движение, возвращает новую клетку, награду и флаг конца (`done`). / The step function defines our world. It takes a cell and action, returning the next cell, reward and an end flag.

**Правила / Rules:** за границу или в стену → остаёмся на месте, −3, False. В цель → +20, True. В обычную клетку → −1, False. Из цели → остаёмся там, 0, True. / Outside the map or into a wall: stay, −3, False. Reach goal: +20, True. Ordinary move: −1, False. Already at goal: stay, 0, True.

Координаты начинаются с нуля; row растёт вниз. Следующий пример только считает возможную клетку, пока не проверяя стены. / Coordinates start at zero; row increases downward. The example only computes a candidate cell without checking walls.

In [ ]:
position = START # Исходная клетка / Starting cell.
dr, dc = MOVES[1] # Смещение вправо / Rightward offset.
candidate = (position[0] + dr, position[1] + dc) # Возможная новая клетка / Candidate next cell.
print('From / из', position, 'to / в', candidate) # Ожидаем (0, 1) / Expect (0, 1).

### Что написать / What to write

Напишите `step(state, action)`. `state` — допустимая свободная клетка; `action` — число 0…3. Дополнительная проверка типов не нужна. / Write step for a valid free cell and an action 0…3; no extra type validation needed.
1. Если state уже GOAL, верните `(state, 0, True)`. / Handle an already reached goal.
2. Возьмите dr, dc из MOVES[action] и вычислите row, col новой клетки. / Compute the candidate coordinates.
3. Если новая клетка за границей или в WALLS, верните прежнее состояние и штраф. / Reject boundaries and walls.
4. Иначе верните новую клетку и награду по правилам выше. / Otherwise return the new cell and the appropriate reward.

In [ ]:
def step(state, action): # Правила одного хода / Rules for one move.
    # TODO: отдельно обработайте GOAL / Handle GOAL first.
    dr, dc = MOVES[action] # Получить смещение / Get the offset.
    row = ... # TODO: новая строка / New row.
    col = ... # TODO: новый столбец / New column.
    # TODO: проверьте границы, WALLS и GOAL / Check boundaries, WALLS and GOAL.
    raise NotImplementedError('Task 1 / Задание 1') # Замените своей реализацией / Replace with your implementation.

### Проверка / Check
Запустите после заполнения функции. / Run after completing the function.

In [ ]:
assert step(START, 1) == ((0, 1), -1, False), 'Right from S / Вправо от S' # Обычный ход / Normal move.
assert step(START, 0) == (START, -3, False), 'Map boundary / Граница карты' # Граница / Boundary.
assert step((0, 1), 1) == ((0, 1), -3, False), 'Wall / Стена' # Столкновение / Collision.
assert step((4, 4), 1) == (GOAL, 20, True), 'Goal reward / Награда цели' # Цель / Goal.
assert step(GOAL, 0) == (GOAL, 0, True), 'Goal is terminal / Цель завершает эпизод' # Завершённое состояние / Terminal state.
print('T1 OK') # Проверки пройдены / Checks passed.

### Короткий ответ / Short answer
Почему штраф за обычный шаг помогает предпочесть короткий путь? / Why does an ordinary-step penalty encourage a shorter route?

**Мой ответ / My answer:** ___

## 2. Выбрать движение / Choose a move

**Зачем / Why:** `choose_action` решает, пробовать новое движение или использовать уже изученное. Q — таблица `(5, 6, 4)`: для каждой клетки четыре оценки. / The function decides whether to explore or use existing knowledge. Q has shape (5, 6, 4), with four estimates per cell.

`epsilon=0.2` означает: примерно в 20% случаев выбрать любое из четырёх действий случайно, иначе взять лучший Q. Случайный выбор тоже может попасть в лучшее действие. При равных Q выбирайте первый максимум. / With epsilon=0.2, approximately 20% of choices are uniform random; otherwise take the highest Q. Random exploration may also select the best action. Resolve ties with the first maximum.

`rng.random()` возвращает число от 0 включительно до 1 не включительно. Проверка `< epsilon` решает, исследовать ли сейчас. / rng.random returns a number in [0,1); comparing it to epsilon decides whether to explore.

In [ ]:
Q_example = np.zeros((ROWS, COLS, 4)) # Пока ничего не знаем / No knowledge yet.
print(Q_example[START]) # Четыре оценки в S / Four estimates at S.
print('First maximum / первый максимум:', np.argmax(Q_example[START])) # При равенстве индекс 0 / Ties select index 0.
demo_rng = np.random.default_rng(7) # Повторяемый пример случайности / Reproducible random example.
print('Random action / случайное действие:', demo_rng.integers(4)) # Одно из 0,1,2,3 / One of 0,1,2,3.

### Что написать / What to write

Напишите `choose_action(Q, state, epsilon, rng)` → целое действие. Все входы корректны. / Write the function returning an integer action; inputs are valid.
1. Получите одно rng.random(). Если оно меньше epsilon, верните int(rng.integers(4)). / Draw once and explore when below epsilon.
2. Иначе верните индекс первого максимума в Q[state]. / Otherwise return the first maximum index in Q[state].
Не меняйте Q и не создавайте новый rng внутри функции. / Do not change Q or create another RNG inside the function.

In [ ]:
def choose_action(Q, state, epsilon, rng): # Выбрать одно действие / Choose one action.
    if rng.random() < epsilon: # Ветка исследования / Exploration branch.
        return ... # TODO: случайное действие / Random action.
    return ... # TODO: действие с максимальным Q / Action with largest Q.

### Проверка / Check
Запустите после заполнения функции. / Run after completing the function.

In [ ]:
probe = np.zeros((ROWS, COLS, 4)) # Проверочная таблица / Probe table.
probe[START] = [-2, 4, 1, 0] # Лучшее действие вправо / Right is best.
before = probe.copy() # Проверить неизменность / Check no mutation.
assert choose_action(probe, START, 0, np.random.default_rng(5)) == 1, 'epsilon=0: take the best / Выберите лучшее' # Без исследования / No exploration.
random_rng = np.random.default_rng(5) # Общий генератор / Shared generator.
actions = [choose_action(probe, START, 1, random_rng) for _ in range(100)] # Полное исследование / Pure exploration.
assert set(actions) == {0, 1, 2, 3}, 'Try all directions / Попробуйте все направления' # Доступны все действия / All actions possible.
assert np.array_equal(probe, before), 'Do not change Q / Не меняйте Q' # Только чтение / Read only.
assert choose_action(np.zeros_like(probe), START, 0, np.random.default_rng(5)) == 0, 'First maximum on ties / Первый максимум при равенстве' # Ничья / Tie.
print('T2 OK') # Проверки пройдены / Checks passed.

### Короткий ответ / Short answer
Что может пойти не так, если с самого начала epsilon=0 и все Q равны нулю? / What can go wrong if epsilon=0 from the start and all Q values are zero?

**Мой ответ / My answer:** ___

## 3. Запомнить результат хода / Learn from a move

**Зачем / Why:** `update_q` меняет одно число в таблице после хода. Мы учитываем награду сейчас **и** возможность получить награду дальше. / Update one table entry after a move, considering both the immediate reward and future opportunities.

**Правило / Rule:** `target = reward + gamma * max(Q[next_state])`, затем / then `new = old + alpha * (target - old)`.

`alpha=0.2`: пройти 20% расстояния от старой оценки до target. `gamma=0.9`: будущая награда имеет меньший вес. Если `done=True`, будущих ходов нет: **target = reward**. / Alpha moves the estimate 20% towards the target. Gamma discounts future reward. If done=True, there are no future moves: target equals reward only.

Это и есть обновление **Q-learning**. Не нужно выводить формулу: переведите её в три строки кода. / This is the Q-learning update. No derivation is required; translate it into three code lines.

In [ ]:
old_value, target_value, learning_rate = 0.0, 20.0, 0.2 # Пример последнего хода в цель / Example last move into goal.
new_value = old_value + learning_rate * (target_value - old_value) # Часть пути к новой оценке / Move partway towards target.
print('Updated estimate / новая оценка:', new_value) # Получится 4, не 20 / Gives 4, not 20.

### Что написать / What to write

Напишите `update_q(Q, state, action, reward, next_state, done, alpha, gamma)`. **Измените Q на месте**; возвращать ничего не требуется. / Modify Q in place; no return value required.
1. Если done, target=reward; иначе добавьте gamma × максимальную оценку следующей клетки. / Form the terminal or nonterminal target.
2. Возьмите old из Q[state][action]. / Read the old entry.
3. Запишите old + alpha × (target − old) в тот же элемент. Остальные элементы не трогайте. / Write the updated value to that entry only.

In [ ]:
def update_q(Q, state, action, reward, next_state, done, alpha, gamma): # Обновить одну оценку / Update one estimate.
    target = ... # TODO: учесть done / Account for done.
    old = Q[state][action] # Текущая оценка / Current estimate.
    Q[state][action] = ... # TODO: шаг от old к target / Step from old towards target.

### Проверка / Check
Запустите после заполнения функции. / Run after completing the function.

In [ ]:
probe = np.zeros((ROWS, COLS, 4)) # Независимый пример / Independent probe.
probe[(0, 1)] = [1, 2, 3, 4] # Лучшее будущее значение 4 / Best next value is 4.
before = probe.copy() # Сохранить остальные элементы / Preserve other entries for comparison.
update_q(probe, START, 1, -1, (0, 1), False, 0.5, 0.9) # Нет конца: target=-1+0.9*4 / Nonterminal target.
assert np.isclose(probe[START][1], 1.3), 'Expected 0.5*(-1+0.9*4) / Проверьте target' # Проверить число / Check numeric result.
before[START][1] = 1.3 # Ожидается одно изменение / Expect one change.
assert np.allclose(probe, before), 'Change one entry only / Меняйте один элемент' # Остальные элементы неизменны / Others unchanged.
probe[GOAL] = 100 # Ловушка: после цели будущей награды нет / Trap: no future reward after goal.
update_q(probe, (4, 4), 1, 20, GOAL, True, 0.5, 0.9) # Последний ход / Terminal move.
assert np.isclose(probe[(4, 4)][1], 10), 'At goal target=reward / У цели target=reward' # Не использовать 100 / Do not use the 100.
print('T3 OK') # Проверки пройдены / Checks passed.

### Короткий ответ / Short answer
Почему нельзя добавлять будущую награду, когда робот уже дошёл до G? / Why must future reward not be added after the robot reaches G?

**Мой ответ / My answer:** ___

## 4. Повторить попытки / Repeat attempts

**Зачем / Why:** `train_robot` соединяет первые три функции. Каждый эпизод начинается в S с уже накопленной Q-таблицей. Внутри: выбрать движение → выполнить ход → обновить Q → перейти в новую клетку. / Combine the first three functions. Every episode starts at S while keeping the learned Q table. Choose → step → update → move to the next cell.

Останавливаем эпизод при цели или после max_steps. Лимит предотвращает бесконечное блуждание. После достижения лимита просто начинаем новую попытку; для обновления флаг done приходит только от step. / Stop at the goal or max_steps to avoid endless wandering. A time limit starts a new attempt; only step supplies the terminal flag for the update.

Каркас уже содержит оба цикла. Нужно соединить ваши функции, а не придумать новый алгоритм. / Both loops are supplied; connect your functions rather than inventing another algorithm.

In [ ]:
next_position, reward, finished = step(START, 1) # Пример одного перехода / Example single transition.
print(next_position, reward, finished) # Клетка, награда, конец / Cell, reward, termination.
print('After the move state becomes next_position / После хода state станет next_position') # Следующий шаг начинается здесь / Next move begins here.

### Что написать / What to write

Заполните четыре места: choose_action, step, update_q и переход state=next_state. / Fill four places: choose_action, step, update_q, and state=next_state.

Верните `(Q, rewards)`: обученная таблица и список суммарных наград по эпизодам. Q создаётся **один раз до цикла эпизодов**, а state сбрасывается в START каждый эпизод. Используйте переданный seed только для одного локального rng. / Return the learned table and episode-reward list. Create Q once before all episodes, reset state each episode, and use one local RNG.

In [ ]:
def train_robot(episodes=500, max_steps=100, epsilon=0.2, alpha=0.2, gamma=0.9, seed=2026): # Повторные попытки / Repeated attempts.
    Q = np.zeros((ROWS, COLS, 4)) # Одна таблица на всё обучение / One table for all training.
    rng = np.random.default_rng(seed) # Один генератор / One generator.
    rewards = [] # Награды эпизодов / Episode rewards.
    for episode in range(episodes): # Начать попытку / Start an attempt.
        state, total = START, 0 # Снова в S, награда пока 0 / Back to S, zero episode reward.
        for move in range(max_steps): # Ограничить число ходов / Limit moves.
            action = ... # TODO 1: вызов choose_action / Call choose_action.
            next_state, reward, done = ... # TODO 2: вызов step / Call step.
            # TODO 3: вызовите update_q с этим переходом / Call update_q on this transition.
            state = ... # TODO 4: перейти в новую клетку / Move to the next cell.
            total += reward # Суммировать награды / Sum rewards.
            if done: # Цель достигнута / Goal reached.
                break # Закончить попытку / End the attempt.
        rewards.append(total) # Сохранить итог попытки / Store episode return.
    return Q, rewards # Готовая таблица и история / Learned table and history.

### Проверка / Check
Запустите после заполнения функции. / Run after completing the function.

In [ ]:
Q_short, rewards_short = train_robot(episodes=3, max_steps=1, epsilon=0, seed=1) # Три короткие попытки / Three one-step attempts.
assert Q_short.shape == (ROWS, COLS, 4) and len(rewards_short) == 3, 'Check output shapes / Проверьте формы' # Формы / Shapes.
assert rewards_short == [-3, -1, -1], 'Keep Q between episodes; reset position / Сохраняйте Q, сбрасывайте позицию' # Порядок движения меняется после обучения / Learning changes later actions.
assert np.allclose(Q_short[START], [-0.6, -0.2, -0.2, 0]), 'Check update and arguments / Проверьте обновление и аргументы' # Численный результат / Numerical result.
Q_a, r_a = train_robot(episodes=20, seed=13) # Первый прогон / First run.
Q_b, r_b = train_robot(episodes=20, seed=13) # Тот же seed / Same seed.
assert np.array_equal(Q_a, Q_b) and r_a == r_b, 'Same seed must reproduce / Seed должен повторять результат' # Повторяемость / Reproducibility.
assert np.isfinite(Q_a).all() and np.count_nonzero(Q_a) > 0, 'Learning must update Q / Обучение должно менять Q' # Обучение состоялось / Updates happened.
print('T4 OK') # Проверки пройдены / Checks passed.

### Короткий ответ / Short answer
Зачем сохранять Q между эпизодами, но возвращать позицию в START? / Why keep Q between episodes but reset the position to START?

**Мой ответ / My answer:** ___

## 5. Показать выученный маршрут / Show the learned route

**Зачем / Why:** во время обучения робот иногда специально ходит случайно. Теперь хотим увидеть, что он выучил. В `run_robot` каждый раз берём лучший Q, ничего не обучаем и сохраняем посещённые клетки. / Training deliberately includes random moves. Now inspect learned behaviour: always take the best Q, never update it, and record visited cells.

Маршрут начинается с START. После каждого хода добавляем next_state, даже если робот остался на месте. Если упёрся в стену или зациклился, лимит ходов остановит запуск. Не скрывайте неудачи. / Start the route with START. Append the next cell after every move, even if unchanged. The move limit stops collisions or loops; do not hide failures.

In [ ]:
example_path = [START, (0, 1), (1, 1)] # Короткий допустимый фрагмент / A short valid fragment.
draw_map(example_path) # Готовый помощник рисует список клеток / Supplied helper draws visited cells.

### Что написать / What to write

Напишите `run_robot(Q, max_steps=30)` → `(path, total_reward, reached_goal)`.
1. Начните с state=START, path=[START], total=0. / Initialise position, path and return.
2. До max_steps раз: выберите int(np.argmax(Q[state])), вызовите step. / Choose the best action and step.
3. Сохраните новую клетку, прибавьте награду; при done прервите цикл. / Save the cell, add reward, and stop if done.
4. Верните маршрут, сумму и bool(state==GOAL). **Не вызывайте update_q.** / Return path, total and goal flag. Do not update Q.

In [ ]:
def run_robot(Q, max_steps=30): # Проверить готовую таблицу / Evaluate learned table.
    state, total = START, 0 # Начало проверки / Evaluation start.
    path = [state] # Маршрут включает старт / Route includes start.
    # TODO: цикл выбора, шага и записи маршрута / Loop over selection, movement and route recording.
    raise NotImplementedError('Task 5 / Задание 5') # Удалите после выполнения / Remove when completed.
    return path, total, state == GOAL # Маршрут, награда, успех / Path, reward, success.

### Проверка / Check
Запустите после заполнения функции. / Run after completing the function.

In [ ]:
untrained = np.zeros((ROWS, COLS, 4)) # Без знаний лучший индекс 0 / Without learning first maximum is 0.
path_zero, total_zero, success_zero = run_robot(untrained, max_steps=3) # Три удара в верхнюю границу / Three upper-boundary collisions.
assert path_zero == [START] * 4 and total_zero == -9 and not success_zero, 'Log collisions and respect limit / Записывайте столкновения и соблюдайте лимит' # Не скрывать неудачу / Preserve failure.
Q_test, _ = train_robot() # Фиксированное обучение / Fixed training.
before = Q_test.copy() # Сохранить веса / Preserve learned table.
path_test, total_test, success_test = run_robot(Q_test) # Проверить маршрут / Evaluate route.
assert np.array_equal(Q_test, before), 'Evaluation must not learn / Проверка не обучает' # Заморозка / Frozen table.
assert path_test[0] == START and len(path_test) <= 31, 'Start and length / Старт и длина' # Контракт маршрута / Path contract.
assert all(0 <= r < ROWS and 0 <= c < COLS and (r, c) not in WALLS for r, c in path_test), 'Stay in free cells / Только свободные клетки' # Допустимые клетки / Valid cells.
assert success_test == (path_test[-1] == GOAL), 'Report actual success / Сообщите реальный успех' # Честный статус / Honest status.
print('T5 OK; reached goal / достиг цели:', success_test) # Высокая награда не заменяет правильный код / High reward does not replace correct code.

### Короткий ответ / Short answer
Чем эта проверка отличается от обучения? Доказывает ли успех на одной карте работу в новом лабиринте? / How does evaluation differ from training? Does success on this map establish success in a new maze?

**Мой ответ / My answer:** ___

## Запустите весь алгоритм / Run the complete algorithm

Это готовая сборка ваших пяти функций. Параметры уже выбраны; менять их для сдачи не нужно. Сначала показан необученный робот, затем обученный. График отражает награды во время случайных проб; финальный маршрут использует только лучшие действия. / This supplied cell combines your five functions with fixed parameters. Compare the untrained and learned robot. The training curve includes exploratory moves; the final route takes only the best actions.

In [ ]:
before_path, before_reward, before_success = run_robot(np.zeros((ROWS, COLS, 4))) # Робот без знаний / Untrained robot.
Q, rewards = train_robot() # Обучить на фиксированной карте / Train on the fixed map.
path, total_reward, reached_goal = run_robot(Q) # Показать поведение без исследования / Evaluate without exploration.
print('Before / до:', before_success, before_reward) # Исходный результат / Initial result.
print('After / после:', reached_goal, total_reward, 'moves / ходов:', len(path) - 1) # Результат обучения / Learned result.
draw_map(path) # Нарисовать выученный маршрут / Draw the learned route.
plt.plot(rewards) # История наград всех попыток / Returns for every attempt.
plt.xlabel('Episode / Попытка'); plt.ylabel('Total reward / Суммарная награда') # Подписи / Labels.
plt.show() # Показать динамику / Display learning curve.

## Посмотрите, как движется робот / Watch the robot move

Слева — робот до обучения, справа — после. Нажмите ▶; ползунок позволяет разобрать каждый ход. Синяя точка — робот, линия — пройденный путь, тёмные клетки — стены, зелёная G — цель. При столкновении робот остаётся на месте, а награда уменьшается. Достигший цели робот останавливается, пока второй заканчивает попытку. / Left: before training; right: after training. Press ▶ or use the slider to inspect each move. The blue dot is the robot, the line its path, dark cells are walls, and green G is the goal. A collision keeps the position unchanged and reduces reward. A robot that reaches the goal stops while the other finishes its attempt.

**Готовая визуализация, не шестое задание.** Выполните её после пяти функций и итогового эксперимента. Она показывает ваши `before_path` и `path`, ничего не обучает. Номер хода, награда последнего хода и накопленная награда выводятся под картами. Если ваш робот не достиг цели, анимация честно покажет это. / **Supplied visualization, not a sixth exercise.** Run it after the five functions and final experiment. It displays your actual paths without training. The move number, latest reward and cumulative reward appear below the maps. Failure to reach the goal is displayed honestly.

В Colab анимация работает прямо в выводе ячейки; отдельные файлы и ffmpeg не нужны. / The animation plays inside Colab output; no separate files or ffmpeg are needed.

In [ ]:
from matplotlib.animation import FuncAnimation # Анимация кадров / Frame animation.
from IPython.display import HTML, display # Проигрыватель внутри Colab / Player inside Colab.
def animate_comparison(paths, titles): # Готовый помощник, не задание / Supplied helper, not an exercise.
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5)) # Две карты рядом / Two maps side by side.
    fig.subplots_adjust(bottom=0.22, top=0.88, wspace=0.3) # Место для подписи наград / Space for reward labels.
    artists, returns = [], [] # Объекты рисунка и накопленные награды / Artists and cumulative returns.
    for ax, route, title in zip(axes, paths, titles): # Настроить каждый маршрут / Set up each route.
        board = np.zeros((ROWS, COLS)) # Свободная карта / Empty map.
        for row, col in WALLS: # Перебрать препятствия / Visit obstacles.
            board[row, col] = 1 # Отметить стену / Mark wall.
        ax.imshow(board, cmap='Greys', vmin=0, vmax=1) # Показать препятствия / Show obstacles.
        ax.set_xticks(np.arange(-0.5, COLS, 1), minor=True) # Вертикальные границы / Vertical cell boundaries.
        ax.set_yticks(np.arange(-0.5, ROWS, 1), minor=True) # Горизонтальные границы / Horizontal cell boundaries.
        ax.grid(which='minor', color='#b8c2cc', linewidth=0.7) # Сетка клеток / Cell grid.
        ax.tick_params(which='minor', bottom=False, left=False) # Скрыть лишние отметки / Hide minor tick marks.
        ax.set_xticks(range(COLS)); ax.set_yticks(range(ROWS)) # Координаты / Coordinates.
        ax.set_xlabel('col / столбец'); ax.set_ylabel('row / строка') # Оси / Axes.
        ax.set_title(title, fontsize=12, fontweight='bold') # До или после обучения / Before or after training.
        ax.text(START[1], START[0], 'S', ha='center', va='center', color='#00804a', fontsize=18) # Старт / Start.
        ax.text(GOAL[1], GOAL[0], 'G', ha='center', va='center', color='#00804a', fontsize=18) # Цель / Goal.
        trail, = ax.plot([], [], '-', color='#4285f4', linewidth=3, alpha=0.55) # След робота / Robot trail.
        dot, = ax.plot([], [], 'o', color='#1557c0', markersize=13, markeredgecolor='white', markeredgewidth=2) # Робот / Robot marker.
        label = ax.text(0.5, -0.24, '', ha='center', va='top', transform=ax.transAxes, fontsize=10) # Текущие показатели / Live metrics.
        rewards_route = [0] # В старте награда ещё не получена / No reward at the start.
        for previous, current in zip(route[:-1], route[1:]): # Прочитать реальные переходы / Read actual transitions.
            reward = 0 if previous == GOAL else 20 if current == GOAL else -3 if current == previous else -1 # Правила среды для записанного пути / Environment rewards for the recorded path.
            rewards_route.append(reward) # Сохранить награду хода / Record move reward.
        returns.append((rewards_route, np.cumsum(rewards_route))) # Награды и их суммы / Rewards and running sums.
        artists.append((trail, dot, label)) # Сохранить изменяемые элементы / Keep animated artists.
    def frame(number): # Показать один момент / Display one instant.
        for route, items, reward_info in zip(paths, artists, returns): # Обновить обе карты / Update both maps.
            index = min(number, len(route) - 1) # Завершившийся робот остаётся на месте / Finished robot stays in place.
            visited = route[:index + 1] # Пройденная часть пути / Visited path prefix.
            trail, dot, label = items # Разобрать элементы / Unpack artists.
            trail.set_data([p[1] for p in visited], [p[0] for p in visited]) # Обновить след / Update trail.
            dot.set_data([route[index][1]], [route[index][0]]) # Передвинуть робота / Move robot marker.
            reached = tuple(route[index]) == GOAL # Проверить реальную цель / Check actual goal arrival.
            status = 'Goal reached / Цель!' if reached else 'Limit reached / Лимит ходов' if index == len(route) - 1 else 'Moving / Движение' # Честный статус / Honest status.
            dot.set_color('#00804a' if reached else '#1557c0') # У цели робот зелёный / Green robot at goal.
            latest, cumulative = reward_info # Награда хода и сумма / Move reward and cumulative return.
            label.set_text(f'Move / Ход: {index} | Reward / Награда: {latest[index]:+d}\nTotal / Сумма: {cumulative[index]:+d} | {status}') # Обновить показатели / Update metrics.
        return [artist for items in artists for artist in items] # Вернуть элементы анимации / Return animated artists.
    animation = FuncAnimation(fig, frame, frames=max(map(len, paths)), interval=450, blit=False, repeat=False) # Один кадр на ход / One frame per move.
    player = HTML(animation.to_jshtml(default_mode='once')) # Встроенный проигрыватель без ffmpeg / Embedded player without ffmpeg.
    plt.close(fig) # Не дублировать статический рисунок / Avoid duplicate static figure.
    return player # Вернуть проигрыватель / Return player.
display(animate_comparison([before_path, path], ['Before training / До обучения', 'After training / После обучения'])) # Сравнить фактические маршруты / Compare actual routes.


## Что сдавать / What to submit

Выполненный `HW10_RL.ipynb`: ФИО и группа, пять функций, пять пройденных проверок, пять коротких ответов, рисунок маршрута и фактические результаты. Перед сдачей Restart & Run all. Отдельный JSON для **этой версии** не нужен. / Submit the completed notebook with name/group, five functions, five passing checks, five short answers, the route plot and actual results. Restart and Run all before submitting. This version requires no separate JSON.

**Результат / Result:** дошёл ли робот / reached goal: ___; число ходов / moves: ___; суммарная награда / return: ___. Что видно на маршруте? / What do you observe in the route? ___

**Источники и помощь / Sources and help:** укажите материалы и использование ИИ; объясните код своими словами. / Acknowledge materials and AI assistance; explain your code in your own words. ___

80 баллов за код: 5 × 16. 20 за объяснения: 5 × 4. Успешный маршрут сам по себе не заменяет корректность функций. Срок и штрафы остаются по матрице группы: raw × max(0,1−просрочка/14 суток), затем ×0.9 при отсутствии отметки P10. / Code: 5 × 16; explanations: 5 × 4. A successful route alone does not establish correct functions. Existing group deadlines and penalties remain: raw × max(0,1−lateness/14 days), then ×0.9 if P10 is unmarked.

**Границы вывода / Limits:** одна маленькая детерминированная карта, точные координаты, нет сенсоров, шума, батареи и физики. Это демонстрация алгоритма, не испытание реального робота. Здесь нет Train/Validation/Test как в прежнем Wine: финальный маршрут проверяется в той же среде и не доказывает перенос на новые карты. / One small deterministic map, exact positions, no sensors, noise, battery or physics. This illustrates an algorithm, not a real robot test. Unlike Wine, there is no train/validation/test split here: the final route uses the training environment and does not establish transfer to new maps.